# Prerana: models and runs (Thu Nov 19)
Track status in the tracker sheet. Each **P** cell = one row in your tab.
Outputs go to the shared Drive folder `lost-at-the-switch/results/`.

### S. Setup (run every time you open a new runtime)
Runtime -> Change runtime type -> **A100**, else **L4** (T4 is too small).
Secrets (key icon, notebook access ON): `GITHUB_TOKEN`, `HF_TOKEN`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, os
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-at-the-switch'
if not os.path.isdir(DRIVE_DIR):
    raise SystemExit("Shared folder not found. In Google Drive: 'Shared with me' -> right-click "
                     "'lost-at-the-switch' -> Organize -> Add shortcut -> My Drive. Then rerun this cell.")
for sub in ['results', 'data/raw', 'data/processed']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HOME'] = '/content/hf_cache'
REPO = '/content/lost-at-the-switch'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull --ff-only
else:
    !git clone -q https://{GITHUB_TOKEN}@github.com/Narpear/lost-at-the-switch.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[dev]"
# data/ and results/ live in the shared Drive folder, so both of us see everything
!rm -rf results data/raw data/processed
!ln -s {DRIVE_DIR}/results results && ln -s {DRIVE_DIR}/data/raw data/raw && ln -s {DRIVE_DIR}/data/processed data/processed
!ls -la results data/raw data/processed | head -20

## P1. Fast kernels + re-run the sanity check
Then copy the new `c` from line [4] into the `C = ...` line in P5. Monday's c=0.69 was with slow fallback kernels.

In [ ]:
# Fast kernels for Qwen3.5's linear-attention layers (results are identical, just faster).
# flash-linear-attention is quick. causal-conv1d may compile for 5-15 min; if it fails, that's OK.
!pip install -q flash-linear-attention 2>&1 | tail -2
!timeout 1200 pip install -q causal-conv1d --no-build-isolation 2>&1 | tail -2
!python -c "import fla; print('fla OK')" ; python -c "import causal_conv1d; print('causal_conv1d OK')"
# If the sanity check below crashes on import, undo with:
#   !pip uninstall -y flash-linear-attention causal-conv1d

In [ ]:
!python scripts/sanity_check.py --pair qwen 2>&1 | grep -v -E "falling back|Loading|Fetching|it/s" | tee results/sanity_qwen_fastkernels.txt

## P2. Llama pair sanity check (only if HF access to Llama-3.2-1B and 3B is approved)

In [ ]:
!python scripts/sanity_check.py --pair llama 2>&1 | grep -v -E "Loading|Fetching|it/s" | tee results/sanity_llama.txt

## P3. Exp 1: teacher-forced acceptance at switch points (needs Manjari's M1 LinCE data)
~10-30 min per pair. Up to 800 code-switched dev sentences each. Skips pairs already done.

In [ ]:
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    pair = cfg.replace('lid_', '')
    print('=' * 20, pair)
    !python scripts/run_positional.py --pair qwen --data data/raw/lince/{cfg}/dev.conll --out results/positional/qwen_{pair}_dev.jsonl --only-code-switched --limit 800 --skip-existing 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -25

## P4. Make code-switched prompts (skipped if Manjari already made them)

In [ ]:
# Code-switched continuation prompts from LinCE dev (first half of each code-switched sentence).
# Same files for both of us; skipped if they already exist.
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    out = f'data/processed/{cfg}_dev_prompts.jsonl'
    if not os.path.exists(out):
        !python scripts/make_prompts.py data/raw/lince/{cfg}/dev.conll {out}
    else:
        print(out, 'exists')

## P5. B1 on code-switched text: Qwen 0.8B drafter, gamma 2/3/4, 50 prompts per pair
Raw continuation prompts (no chat template). Resumes automatically if Colab disconnects: just rerun the cell.

In [ ]:
C = None   # <- paste c from P1, e.g. C = 0.12
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    pair = cfg.replace('lid_', '')
    print('=' * 20, pair)
    !python scripts/run_specdec.py --pair qwen --prompts data/processed/{cfg}_dev_prompts.jsonl --out results/specdec/qwen_b1_{pair}.jsonl --limit 50 --gammas 2 3 4 --skip-existing {'--c ' + str(C) if C else ''} 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -30

## P6. B1 reproduction on Nepali (overnight): FLORES English->Nepali translation, chat template
Needs Manjari's M2 FLORES prompts. Paudel's own Nepali test set isn't released, so FLORES is our stand-in.

In [ ]:
!python scripts/run_specdec.py --pair qwen --prompts data/processed/flores_translate_npi_Deva.jsonl --out results/specdec/qwen_b1_flores_en2ne.jsonl --limit 100 --gammas 2 3 4 --chat-template --skip-existing 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -30

## P7. Tables + figure for Friday

In [ ]:
!python scripts/make_report.py
from IPython.display import Image, display
if os.path.exists('results/report/positional.png'): display(Image('results/report/positional.png'))